## Epic 2 — Preprocessing Pipeline

Transforms the raw `MetObjects.csv` into a clean, model-ready feature matrix.
Covers null handling, column selection, and feature engineering for both the supervised (Department classification) and unsupervised (clustering) tracks.

In [1]:
import os

import numpy as np
import pandas as pd
import scipy.sparse
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder

In [2]:
ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
DATA_PATH = os.path.join(ROOT, "data", "MetObjects.csv")

# Setting low_memory=False forces pandas to read the entire column before deciding its dtype, which is slower but correct.
df_raw = pd.read_csv(DATA_PATH, low_memory=False)

print("Loaded:", df_raw.shape)

Loaded: (484956, 54)


In [3]:
# ── Column definitions (single source of truth) ────────────────────────────
# Source: reports/feature_engineering.md

COLS_TO_DROP = [
    # Identifiers
    "Object ID", "Object Number", "Constituent ID",
    # URLs — no feature signal
    "Link Resource", "Artist ULAN URL", "Artist Wikidata URL",
    "Object Wikidata URL", "Tags AAT URL", "Tags Wikidata URL",
    # 100% null / constant
    "Metadata Date", "Repository",
    # Geographic columns >90% null
    "River", "State", "Locus", "County", "Locale",
    "Excavation", "Subregion", "Region", "City",
    # Admin fields with no object-level signal
    "Rights and Reproduction", "Portfolio", "Gallery Number", "Credit Line",
    # Redundant / low-signal artist fields
    "Artist Prefix", "Artist Suffix", "Artist Alpha Sort",
    "Artist Display Bio", "Artist Display Name",
    # Flagged — dropping for now
    "Dynasty", "Reign", "Geography Type", "Country",
    "Artist Begin Date", "Artist End Date",
    "Dimensions", "Title", "Object Date",
]

COLS_NUMERIC = [
    "AccessionYear",          # 0.8% null → fill with median year
]

COLS_CATEGORICAL = [
    "Medium",                 # 1.5% null  — TF-IDF after comma-splitting
    "Tags",                   # 60.3% null — TF-IDF after pipe-splitting
    "Classification",         # 16.2% null — TF-IDF or top-N encode
    "Object Name",            # 0.5% null  — top-N frequency encode
    "Culture",                # 57.1% null — top-100 + "Other"
    "Period",                 # 81.2% null — top-N + "Other"
    "Artist Nationality",     # 41.7% null — top-N + "Other"
    "Artist Role",            # 41.7% null — label encode
    "Artist Gender",          # 78.0% null — binary flag
]

COLS_KEEP = [
    "Department",             # classification target — kept, encoded separately
    "Object Begin Date",      # int64, 0% null — clip + derive features
    "Object End Date",        # int64, 0% null — clip + derive features
    "Is Highlight",           # bool, 0% null
    "Is Timeline Work",       # bool, 0% null
    "Is Public Domain",       # bool, 0% null
]

print("Columns accounted for:",
      len(COLS_TO_DROP) + len(COLS_NUMERIC) + len(COLS_CATEGORICAL) + len(COLS_KEEP),
      "/ 54 total")

Columns accounted for: 54 / 54 total


In [6]:
df = df_raw.drop(columns=COLS_TO_DROP)

print(f"Shape before drop : {df_raw.shape}")
print(f"Shape after drop  : {df.shape}")
print(f"Columns removed   : {df_raw.shape[1] - df.shape[1]}")
print()
assert "Department" in df.columns, "ERROR: Department was accidentally dropped!"
print("Test - Department column intact!!")

Shape before drop : (484956, 54)
Shape after drop  : (484956, 16)
Columns removed   : 38

Test - Department column intact!!


### Missing Value Imputation

In [7]:
# Null counts before imputation — only show columns that have any nulls
null_before = df.isna().sum()
null_before = null_before[null_before > 0].sort_values(ascending=False)

print(f"Columns with nulls: {len(null_before)}")
print(f"Total null cells  : {null_before.sum():,}")
print()
display(null_before.rename("null_count").to_frame())

Columns with nulls: 10
Total null cells  : 1,838,500



,null_count
Period,393813
Artist Gender,378474
Tags,292501
Culture,276766
Artist Role,202443
Artist Nationality,202443
Classification,78717
Medium,7215
AccessionYear,3862
Object Name,2266


In [9]:
COLS_BOOL = ["Is Highlight", "Is Public Domain", "Is Timeline Work"]

# Numeric: coerce to float first (AccessionYear is stored as string in the CSV),
# then fill nulls introduced by coercion with the column median
for col in COLS_NUMERIC:
    df[col] = pd.to_numeric(df[col], errors="coerce")
    df[col] = df[col].fillna(df[col].median())

# Categorical: fill with "Unknown" so the model sees a real category, not NaN
for col in COLS_CATEGORICAL:
    df[col] = df[col].fillna("Unknown")

# Booleans: fill with False (absence of a flag is the safe neutral value)
for col in COLS_BOOL:
    df[col] = df[col].fillna(False)

### Notes

- AccessionYear is the only numeric column here,
  and year values are likely slightly skewed (e.g.
   a burst of acquisitions in a particular era).
  Mean is pulled by outliers — if a handful of
  records have a weird year like 1800 or 2050, the
   mean shifts. Median is the middle value of the
  sorted column, so outliers can't distort it.
- Top-N frequency encode / label encode:
  "Unknown" becomes its own category. The model
  learns a weight for it — effectively learning "objects with no Culture info tend to land in
  these departments." That's genuinely useful
  signal, because missing = anonymous object is
  not random noise.

In [14]:
remaining_nulls = df.isna().sum().sum()
print(f"Total null cells after imputation: {remaining_nulls}")
assert remaining_nulls == 0, f"Expected 0 nulls, found {remaining_nulls}"
print("Test -- All nulls resolved!")

Total null cells after imputation: 0
Test -- All nulls resolved!
